# VESTIGIUM | Data Engineering Walkthrough

**NASA Space Apps Challenge 2026 · Team Ga Tidur**

This notebook reads the **outputs** from `build_pipeline.py`. It does not re-process 6 million CSV rows in-memory. It verifies contracts, explores lineage, and demonstrates the query shapes required by the map and temporal dashboard. All dates are UTC; all recorded fires are satellite *detections*, not confirmed fire events.

## 1. Setup and reproducibility

Run the pipeline first:

```bash
python build_pipeline.py --input ../data.zip --output ./outputs
```

Run the pipeline from repository root: `python src/pipelines/build_pipeline.py --input data/raw/data.zip --output data/processed/data-engineering`. This notebook reads outputs in `data/processed/data-engineering/`.

In [ ]:
from pathlib import Path
import sqlite3, json, csv
import pandas as pd
import matplotlib.pyplot as plt

import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'README.md').exists() and (p/'data').is_dir()), None)
assert ROOT is not None, 'Open this notebook from within the VESTIGIUM repository'
sys.path.insert(0, str(ROOT/'src'/'pipelines'))
DATA_DIR = ROOT / 'data' / 'processed' / 'data-engineering'
DB = DATA_DIR / 'vestigium_hotspots.sqlite'
QA = json.loads((DATA_DIR / 'quality_report.json').read_text())
assert DB.exists(), 'Run build_pipeline.py first'
print('Archive SHA-256:', QA['input_sha256'])
print('Detected rows:', QA['accepted_presumed_vegetation_detections'])
print('Skipped non-vegetation:', QA['excluded_non_vegetation_detections'])
print('Rejected invalid rows:', QA['rejected_invalid_rows'])

## 2. Data provenance, schema, and sensor inventory

All input CSVs are hashed and retained in `source_files`. Satellite confidence is intentionally split into `confidence_modis_pct` and `confidence_viirs_label`.

In [ ]:
con=sqlite3.connect(f'file:{DB.resolve()}?mode=ro', uri=True)
files=pd.read_sql_query("""SELECT sensor, nominal_year, filename, source_rows, accepted_type0, excluded_non_vegetation,
 rejected_invalid, first_date_utc, last_date_utc FROM source_files ORDER BY sensor,nominal_year""",con)
display(files.head(8))
print('Input files:', len(files))
print(files.groupby('sensor')[['source_rows','accepted_type0','excluded_non_vegetation','rejected_invalid']].sum())

## 3. Full quality gate reconciliation

Counts must reconcile across the ingestion ledger, the individual points table and each summary. This guards against silent row multiplication or dropping rows during grouping.

In [ ]:
db_count=con.execute('SELECT COUNT(*) FROM hotspots').fetchone()[0]
assert db_count == QA['accepted_presumed_vegetation_detections']
for filename in ['daily_sensor.csv','monthly_sensor.csv','annual_sensor.csv','monthly_grid.csv','annual_grid.csv']:
    table=pd.read_csv(DATA_DIR/filename)
    assert int(table['detection_count'].sum()) == db_count, filename
    print(filename, len(table), 'groups; reconciles:', int(table['detection_count'].sum()))
print('PASS: all five groups reconcile with', db_count, 'detection rows')

## 4. Annual detection history, separated by sensor

These are **not** unique fire events and their absolute counts are not directly comparable across sensors, because spatial resolution, observing conditions and overpass frequency differ.

In [ ]:
annual=pd.read_csv(DATA_DIR/'annual_sensor.csv')
pivot=annual.pivot(index='year',columns='sensor',values='detection_count')
ax=pivot.plot(figsize=(11,4),marker='o',linewidth=1.5)
ax.set(title='Annual satellite fire detections by sensor (unharmonized)', ylabel='Detected pixels',xlabel='UTC year')
plt.tight_layout();plt.show()

## 5. Monthly history and seasonality

Missing month rows may mean no detections or incomplete observations. The pipeline does **not** invent zeros in the source.

In [ ]:
monthly=pd.read_csv(DATA_DIR/'monthly_sensor.csv')
sample=monthly[monthly['year']==2015]
ax=sample.pivot(index='month',columns='sensor',values='detection_count').plot(kind='line',marker='o',figsize=(10,4))
ax.set(title='2015 monthly detection activity',xlabel='UTC month',ylabel='Detected pixels')
plt.tight_layout();plt.show()

## 6. Grid aggregation sanity check

The map grid is fixed at **0.25 degrees**. Center points are display locations for groups and must not be mistaken for a detected fire's exact pixel centroid.

In [ ]:
grid=pd.read_csv(DATA_DIR/'annual_grid.csv')
grid2015=grid[(grid.year==2015)&(grid.sensor=='MODIS')]
print('Active 2015 MODIS cells:', len(grid2015))
print('Annual total matches:',grid2015.detection_count.sum(),annual.query("sensor=='MODIS' and year==2015").detection_count.iloc[0])
ax=grid2015.plot.scatter(x='grid_center_lon',y='grid_center_lat',s=grid2015.detection_count.clip(1,1000)/30,alpha=.4,figsize=(10,4))
ax.set(title='2015 MODIS grid centers: sample density view (not individual fire points)',xlabel='Longitude',ylabel='Latitude')
plt.tight_layout();plt.show()

## 7. Individual detections for hover payloads

Spatially restricted SQL protects the browser from millions of points. The dashboard should use clustering or grid aggregates when zoomed out.

In [ ]:
from example_queries import fetch_hotspots
spots=fetch_hotspots(str(DB),2015,9,108,-4,119,4,limit=10,sensor='MODIS')
display(pd.DataFrame(spots))

## 8. Sensor confidence semantic checks

MODIS uses a numerical 0–100 confidence scale; VIIRS uses categorical `l/n/h`. Mixing them as a single numerical predictor would be scientifically unsound.

In [ ]:
assert con.execute("SELECT COUNT(*) FROM hotspots WHERE sensor='MODIS' AND confidence_viirs_label IS NOT NULL").fetchone()[0] == 0
assert con.execute("SELECT COUNT(*) FROM hotspots WHERE sensor LIKE 'VIIRS%' AND confidence_modis_pct IS NOT NULL").fetchone()[0] == 0
print('Sensor-specific confidence storage: PASS')

## 9. Connection to NASA harmonization and social impact module

This ETL is **not** sensor harmonization. It provides provenance, temporal records, spatial bins and a stable query substrate. The next step is comparing overlapping MODIS/VIIRS observation windows with coverage corrections and held-out evaluation. The data do **not** provide trustworthy labels for intentional vs natural ignition.

For long-term land changes, add distinct area-burned and land-cover products, including spatial overlays and documented time-alignment uncertainty. Do not treat a hotspot point as a burned-area polygon.

In [ ]:
con.close()
print('Completed VESTIGIUM Data Engineering walkthrough.')